# Atividade 1
# Agente de Triagem de Manutenção Técnica Industrial

## Contexto

Em um ambiente industrial, solicitações de manutenção podem ser abertas por diferentes setores e apresentar níveis variados de detalhamento, urgência e criticidade. Essas solicitações podem envolver equipamentos produtivos, instalações elétricas, sistemas mecânicos, infraestrutura predial e outros recursos necessários à operação.

Antes do atendimento pela equipe de manutenção, é necessário realizar uma triagem inicial para compreender o problema relatado, avaliar sua prioridade e identificar quais informações adicionais devem ser consultadas.


## Problema

A triagem manual de solicitações de manutenção exige interpretação das descrições fornecidas pelos usuários e consulta a diferentes fontes de informação. Dependendo do caso, pode ser necessário verificar procedimentos de segurança, histórico do equipamento, criticidade da ocorrência ou equipe responsável.

Como as solicitações são descritas em linguagem natural e podem apresentar situações bastante diferentes, o processo não segue necessariamente uma sequência fixa de etapas.


## Objetivo do agente

Desenvolver um agente baseado em LLM capaz de realizar a triagem inicial de solicitações de manutenção técnica industrial, apoiando a equipe responsável na análise e no encaminhamento dos chamados.

O agente deverá avaliar, quando aplicável:

- tipo de problema;
- criticidade;
- urgência;
- risco de interrupção da operação;
- necessidade de intervenção imediata;
- equipe responsável;
- procedimentos aplicáveis;
- existência de histórico relacionado ao equipamento.


## Usuário do sistema

O sistema será utilizado como apoio à equipe responsável pela triagem e encaminhamento das solicitações de manutenção.


## Entradas

A principal entrada será uma solicitação de manutenção descrita em linguagem natural, contendo informações como equipamento envolvido, sintomas observados, localização e impacto percebido na operação.


## Saídas

O agente deverá produzir uma análise estruturada contendo, conforme o caso:

- classificação da solicitação;
- nível de prioridade;
- informações e evidências consultadas;
- equipe ou área sugerida para atendimento;
- recomendação de encaminhamento.

O agente será utilizado como sistema de apoio à decisão, não substituindo a avaliação final dos profissionais responsáveis.


## Justificativa para abordagem agêntica

A abordagem agêntica é adequada porque o agente precisa interpretar solicitações em linguagem natural, avaliar o contexto de cada ocorrência e decidir dinamicamente quais informações devem ser consultadas antes de produzir uma recomendação.

Nem todas as solicitações exigem as mesmas verificações. Um problema simples pode ser classificado diretamente, enquanto uma ocorrência envolvendo risco operacional pode exigir consulta a procedimentos de segurança, histórico do equipamento ou outras fontes de informação.

Dessa forma, o fluxo de processamento não pode ser definido apenas como uma sequência determinística fixa. O agente deverá decidir quais ações executar com base nas informações disponíveis e nos resultados obtidos durante a análise.
## Arquitetura inicial da solução

A solução será implementada inicialmente como um sistema de agente único baseado em LLM. O agente receberá a descrição de uma solicitação de manutenção e deverá interpretar o problema, identificar quais informações adicionais são necessárias, selecionar as ferramentas adequadas e utilizar os resultados obtidos para produzir uma recomendação de triagem.

O fluxo seguirá um ciclo inspirado no padrão ReAct:

**solicitação → decisão → ação → observação → nova decisão → resposta final**

A cada etapa, o agente poderá concluir que já possui informações suficientes ou solicitar uma ferramenta adicional.


### Componentes principais

**1. Entrada da solicitação**

O usuário fornecerá uma descrição em linguagem natural do problema observado, incluindo, quando disponíveis, informações sobre equipamento, localização, sintomas e impacto na operação.

**2. Agente baseado em LLM**

O LLM será responsável por interpretar a solicitação, avaliar quais informações estão ausentes, selecionar as ferramentas apropriadas e integrar as evidências obtidas para produzir a análise final.

**3. Base de conhecimento**

Uma pequena base documental conterá procedimentos e orientações relacionados à manutenção e à segurança. O acesso será realizado por meio de recuperação de informação, permitindo ao agente consultar apenas os trechos relevantes para cada ocorrência.

**4. Ferramentas**

O agente terá inicialmente acesso às seguintes ferramentas:

- `consultar_procedimentos(tipo_problema)`: recupera procedimentos e orientações aplicáveis à ocorrência;
- `consultar_historico_equipamento(equipamento)`: consulta registros anteriores de manutenção e falhas do equipamento;
- `consultar_dados_equipamento(equipamento)`: recupera informações como tipo, localização, criticidade, estado operacional e impacto de eventual parada;
- `consultar_disponibilidade_equipe(especialidade)`: consulta a disponibilidade da equipe técnica adequada ao atendimento.

**5. Runtime**

O runtime será responsável por controlar o ciclo de execução, validar as chamadas das ferramentas, executar as funções correspondentes, devolver os resultados ao agente e aplicar limites de execução e tratamento de erros.

**6. Saída**

Após reunir evidências suficientes, o agente deverá produzir uma resposta estruturada contendo a classificação da ocorrência, prioridade sugerida, evidências relevantes, área técnica recomendada e encaminhamento sugerido.

O agente funcionará como apoio à triagem, sem substituir a decisão dos profissionais responsáveis pela manutenção.


## Preparação do ambiente

In [ ]:
%pip install -q pandas chromadb google-genai

# Carregamento e inspeção dos dados

In [2]:
import pandas as pd
from pathlib import Path

In [3]:
print(pd.__version__)

3.0.6


In [5]:
BASE_DIR = Path(".")

arquivo_equipamentos = BASE_DIR / "equipamentos.csv"
arquivo_historico = BASE_DIR / "historico_manutencao.csv"
arquivo_equipes = BASE_DIR / "equipes.csv"

documentos = [
    BASE_DIR / "procedimento_seguranca.txt",
    BASE_DIR / "procedimento_mecanico.txt",
    BASE_DIR / "procedimento_eletrico.txt",
    BASE_DIR / "politica_prioridades.txt",
]

In [6]:
equipamentos = pd.read_csv(arquivo_equipamentos)
historico = pd.read_csv(arquivo_historico)
equipes = pd.read_csv(arquivo_equipes)

In [7]:
print("Equipamentos:")
display(equipamentos)

print("\nHistórico de manutenção:")
display(historico)

print("\nEquipes:")
display(equipes)

Equipamentos:


,id_equipamento,nome,tipo,setor,criticidade,status,impacto_parada
0,M-204,Motor Principal Linha 2,Motor elétrico,Linha 2,alta,operando,Interrompe completamente a Linha 2
1,M-105,Motor Esteira Linha 1,Motor elétrico,Linha 1,media,operando,Reduz a capacidade da Linha 1
2,P-301,Bomba de Refrigeração,Bomba centrífuga,Utilidades,alta,operando,Pode causar interrupção de equipamentos depend...
3,C-101,Compressor Principal,Compressor,Utilidades,alta,operando,Afeta o fornecimento de ar comprimido para a p...
4,Q-201,Quadro Elétrico Linha 2,Quadro elétrico,Linha 2,alta,operando,Pode interromper toda a alimentação elétrica d...
5,V-401,Ventilador Almoxarifado,Ventilador,Almoxarifado,baixa,operando,Baixo impacto operacional
6,AC-102,Ar-condicionado Administrativo,Ar-condicionado,Administrativo,baixa,operando,Sem impacto direto na produção
7,L-501,Luminária Almoxarifado,Iluminação,Almoxarifado,baixa,operando,Baixo impacto operacional



Histórico de manutenção:


,id_evento,id_equipamento,data,tipo_ocorrencia,descricao,acao_realizada
0,H001,M-204,2026-08-12,Vibracao,Vibração acima do comportamento habitual,Inspeção e reaperto da base
1,H002,M-204,2026-08-15,Rolamento,Ruído e desgaste identificado no rolamento,Troca do rolamento
2,H003,M-204,2026-10-03,Aquecimento,Temperatura elevada durante operação,Inspeção e limpeza do sistema de ventilação
3,H004,M-105,2026-07-20,Ruido,Ruído intermitente durante operação,Inspeção sem falha crítica identificada
4,H005,P-301,2026-09-02,Vazamento,Pequeno vazamento próximo à vedação,Troca da vedação
5,H006,C-101,2026-06-18,Pressao,Queda ocasional de pressão,Limpeza do filtro e inspeção
6,H007,Q-201,2026-09-25,Desarme,Desarme inesperado do disjuntor,Inspeção elétrica e reaperto de conexões
7,H008,AC-102,2026-05-10,Refrigeracao,Equipamento com baixa capacidade de refrigeração,Limpeza de filtros



Equipes:


,id_equipe,nome,especialidade,disponivel,tempo_estimado_minutos
0,E01,Equipe Elétrica,eletrica,sim,20
1,E02,Equipe Mecânica,mecanica,sim,30
2,E03,Equipe Utilidades,utilidades,nao,120
3,E04,Equipe Predial,predial,sim,60
4,E05,Equipe Refrigeração,refrigeracao,sim,45


In [8]:
conteudo_documentos = {}

for caminho in documentos:
    with open(caminho, "r", encoding="utf-8") as f:
        conteudo_documentos[caminho.name] = f.read()

In [9]:
for nome, texto in conteudo_documentos.items():
    print("=" * 60)
    print(nome)
    print("=" * 60)
    print(texto[:500])
    print()

procedimento_seguranca.txt
PROCEDIMENTO DE SEGURANÇA EM MANUTENÇÃO

Ocorrências envolvendo cheiro de queimado, presença de fumaça, faíscas,
sobreaquecimento anormal de componentes elétricos ou desarmes repetidos
devem ser tratadas como situações potencialmente críticas.

Quando houver indício de risco elétrico, a equipe responsável deverá ser
acionada com prioridade. O equipamento não deverá ser operado quando houver
evidência de condição insegura.

Situações com possibilidade de dano a pessoas ou equipamentos devem receb

procedimento_mecanico.txt
PROCEDIMENTO DE MANUTENÇÃO MECÂNICA

Vibração excessiva, ruídos anormais, aquecimento e alterações no comportamento
de motores, bombas e equipamentos rotativos devem ser avaliados considerando
a criticidade do equipamento e seu histórico recente.

A repetição de ocorrências semelhantes pode indicar degradação progressiva
e deve ser considerada na priorização da inspeção.

Equipamentos cuja parada interrompa diretamente o processo produtivo d

In [10]:
print("Quantidade de equipamentos:", len(equipamentos))
print("Quantidade de eventos históricos:", len(historico))
print("Quantidade de equipes:", len(equipes))

Quantidade de equipamentos: 8
Quantidade de eventos históricos: 8
Quantidade de equipes: 5


In [11]:
print(equipamentos.dtypes)
print(historico.dtypes)
print(equipes.dtypes)

id_equipamento    str
nome              str
tipo              str
setor             str
criticidade       str
status            str
impacto_parada    str
dtype: object
id_evento          str
id_equipamento     str
data               str
tipo_ocorrencia    str
descricao          str
acao_realizada     str
dtype: object
id_equipe                   str
nome                        str
especialidade               str
disponivel                  str
tempo_estimado_minutos    int64
dtype: object


# Ferramentas operacionais

In [12]:
def consultar_dados_equipamento(id_equipamento: str) -> dict:
    resultado = equipamentos[
        equipamentos["id_equipamento"].str.upper() == id_equipamento.upper()
    ]

    if resultado.empty:
        return {
            "sucesso": False,
            "erro": "equipamento_nao_encontrado",
            "id_equipamento": id_equipamento
        }

    registro = resultado.iloc[0]

    return {
        "sucesso": True,
        "id_equipamento": registro["id_equipamento"],
        "nome": registro["nome"],
        "tipo": registro["tipo"],
        "setor": registro["setor"],
        "criticidade": registro["criticidade"],
        "status": registro["status"],
        "impacto_parada": registro["impacto_parada"]
    }

In [13]:
consultar_dados_equipamento("M-204")

{'sucesso': True,
 'id_equipamento': 'M-204',
 'nome': 'Motor Principal Linha 2',
 'tipo': 'Motor elétrico',
 'setor': 'Linha 2',
 'criticidade': 'alta',
 'status': 'operando',
 'impacto_parada': 'Interrompe completamente a Linha 2'}

Caso inexistente:

In [14]:
consultar_dados_equipamento("X-999")

{'sucesso': False,
 'erro': 'equipamento_nao_encontrado',
 'id_equipamento': 'X-999'}

In [15]:
def consultar_historico_equipamento(id_equipamento: str) -> dict:
    resultado = historico[
        historico["id_equipamento"].str.upper() == id_equipamento.upper()
    ]

    if resultado.empty:
        return {
            "sucesso": True,
            "id_equipamento": id_equipamento,
            "quantidade_eventos": 0,
            "eventos": []
        }

    eventos = resultado.to_dict(orient="records")

    return {
        "sucesso": True,
        "id_equipamento": id_equipamento,
        "quantidade_eventos": len(eventos),
        "eventos": eventos
    }

In [16]:
consultar_historico_equipamento("M-204")

{'sucesso': True,
 'id_equipamento': 'M-204',
 'quantidade_eventos': 3,
 'eventos': [{'id_evento': 'H001',
   'id_equipamento': 'M-204',
   'data': '2026-08-12',
   'tipo_ocorrencia': 'Vibracao',
   'descricao': 'Vibração acima do comportamento habitual',
   'acao_realizada': 'Inspeção e reaperto da base'},
  {'id_evento': 'H002',
   'id_equipamento': 'M-204',
   'data': '2026-08-15',
   'tipo_ocorrencia': 'Rolamento',
   'descricao': 'Ruído e desgaste identificado no rolamento',
   'acao_realizada': 'Troca do rolamento'},
  {'id_evento': 'H003',
   'id_equipamento': 'M-204',
   'data': '2026-10-03',
   'tipo_ocorrencia': 'Aquecimento',
   'descricao': 'Temperatura elevada durante operação',
   'acao_realizada': 'Inspeção e limpeza do sistema de ventilação'}]}

Equipamento sem histórico:

In [17]:
consultar_historico_equipamento("L-501")

{'sucesso': True,
 'id_equipamento': 'L-501',
 'quantidade_eventos': 0,
 'eventos': []}

In [18]:
def consultar_disponibilidade_equipe(especialidade: str) -> dict:
    resultado = equipes[
        equipes["especialidade"].str.lower() == especialidade.lower()
    ]

    if resultado.empty:
        return {
            "sucesso": False,
            "erro": "especialidade_nao_encontrada",
            "especialidade": especialidade
        }

    registros = resultado.to_dict(orient="records")

    return {
        "sucesso": True,
        "especialidade": especialidade,
        "equipes": registros
    }

In [19]:
consultar_disponibilidade_equipe("mecanica")

{'sucesso': True,
 'especialidade': 'mecanica',
 'equipes': [{'id_equipe': 'E02',
   'nome': 'Equipe Mecânica',
   'especialidade': 'mecanica',
   'disponivel': 'sim',
   'tempo_estimado_minutos': 30}]}

In [20]:
consultar_disponibilidade_equipe("utilidades")

{'sucesso': True,
 'especialidade': 'utilidades',
 'equipes': [{'id_equipe': 'E03',
   'nome': 'Equipe Utilidades',
   'especialidade': 'utilidades',
   'disponivel': 'nao',
   'tempo_estimado_minutos': 120}]}

In [21]:
TOOLS = {
    "consultar_dados_equipamento": consultar_dados_equipamento,
    "consultar_historico_equipamento": consultar_historico_equipamento,
    "consultar_disponibilidade_equipe": consultar_disponibilidade_equipe,
}

# Base de conhecimento e Vector RAG

Os procedimentos e políticas de manutenção constituem conhecimento documental.
Esses documentos serão divididos em trechos e indexados para permitir recuperação
semântica sob demanda pelo agente.

In [22]:
def criar_chunks_documentos(conteudo_documentos: dict) -> list:
    chunks = []

    for nome_arquivo, texto in conteudo_documentos.items():

        paragrafos = [
            paragrafo.strip()
            for paragrafo in texto.split("\n\n")
            if paragrafo.strip()
        ]

        for indice, paragrafo in enumerate(paragrafos):
            chunks.append({
                "id": f"{nome_arquivo}_{indice}",
                "documento": nome_arquivo,
                "texto": paragrafo
            })

    return chunks


chunks = criar_chunks_documentos(conteudo_documentos)

print("Quantidade de chunks:", len(chunks))

Quantidade de chunks: 17


In [23]:
for chunk in chunks:
    print(chunk)
    print("-" * 60)

{'id': 'procedimento_seguranca.txt_0', 'documento': 'procedimento_seguranca.txt', 'texto': 'PROCEDIMENTO DE SEGURANÇA EM MANUTENÇÃO'}
------------------------------------------------------------
{'id': 'procedimento_seguranca.txt_1', 'documento': 'procedimento_seguranca.txt', 'texto': 'Ocorrências envolvendo cheiro de queimado, presença de fumaça, faíscas,\nsobreaquecimento anormal de componentes elétricos ou desarmes repetidos\ndevem ser tratadas como situações potencialmente críticas.'}
------------------------------------------------------------
{'id': 'procedimento_seguranca.txt_2', 'documento': 'procedimento_seguranca.txt', 'texto': 'Quando houver indício de risco elétrico, a equipe responsável deverá ser\nacionada com prioridade. O equipamento não deverá ser operado quando houver\nevidência de condição insegura.'}
------------------------------------------------------------
{'id': 'procedimento_seguranca.txt_3', 'documento': 'procedimento_seguranca.txt', 'texto': 'Situações com p

In [24]:
import chromadb

In [25]:
chroma_client = chromadb.Client()

collection = chroma_client.get_or_create_collection(
    name="procedimentos_manutencao"
)

In [26]:
ids = [chunk["id"] for chunk in chunks]
textos = [chunk["texto"] for chunk in chunks]

metadados = [
    {"documento": chunk["documento"]}
    for chunk in chunks
]

In [27]:
collection.add(
    ids=ids,
    documents=textos,
    metadatas=metadados
)

/Users/lucasminelvino/.cache/chroma/onnx_models/all-MiniLM-L6-v2/onnx.tar.gz: 100%|██████████| 79.3M/79.3M [00:05<00:00, 15.2MiB/s]


In [28]:
print("Itens indexados:", collection.count())

Itens indexados: 17


In [29]:
len(chunks)

17

In [30]:
resultado = collection.query(
    query_texts=["motor apresentando vibração forte e aquecimento"],
    n_results=3
)

resultado

{'ids': [['procedimento_mecanico.txt_1',
   'procedimento_eletrico.txt_1',
   'procedimento_mecanico.txt_3']],
 'embeddings': None,
 'documents': [['Vibração excessiva, ruídos anormais, aquecimento e alterações no comportamento\nde motores, bombas e equipamentos rotativos devem ser avaliados considerando\na criticidade do equipamento e seu histórico recente.',
   'Falhas envolvendo quadros elétricos, disjuntores, motores, cabos e sistemas de\nalimentação devem ser encaminhadas à equipe elétrica quando houver indícios\nde origem elétrica.',
   'Equipamentos cuja parada interrompa diretamente o processo produtivo devem\nreceber prioridade superior aos equipamentos sem impacto direto na produção.']],
 'uris': None,
 'included': ['metadatas', 'documents', 'distances'],
 'data': None,
 'metadatas': [[{'documento': 'procedimento_mecanico.txt'},
   {'documento': 'procedimento_eletrico.txt'},
   {'documento': 'procedimento_mecanico.txt'}]],
 'distances': [[0.5453200340270996, 1.030877113342285

In [31]:
for documento, metadata in zip(
    resultado["documents"][0],
    resultado["metadatas"][0]
):
    print("Fonte:", metadata["documento"])
    print("Trecho:", documento)
    print("-" * 60)

Fonte: procedimento_mecanico.txt
Trecho: Vibração excessiva, ruídos anormais, aquecimento e alterações no comportamento
de motores, bombas e equipamentos rotativos devem ser avaliados considerando
a criticidade do equipamento e seu histórico recente.
------------------------------------------------------------
Fonte: procedimento_eletrico.txt
Trecho: Falhas envolvendo quadros elétricos, disjuntores, motores, cabos e sistemas de
alimentação devem ser encaminhadas à equipe elétrica quando houver indícios
de origem elétrica.
------------------------------------------------------------
Fonte: procedimento_mecanico.txt
Trecho: Equipamentos cuja parada interrompa diretamente o processo produtivo devem
receber prioridade superior aos equipamentos sem impacto direto na produção.
------------------------------------------------------------


In [32]:
def consultar_procedimentos(consulta: str, n_resultados: int = 3) -> dict:

    if not consulta or not consulta.strip():
        return {
            "sucesso": False,
            "erro": "consulta_vazia"
        }

    try:
        resultado = collection.query(
            query_texts=[consulta],
            n_results=n_resultados
        )

        documentos_encontrados = []

        for texto, metadata in zip(
            resultado["documents"][0],
            resultado["metadatas"][0]
        ):
            documentos_encontrados.append({
                "documento": metadata["documento"],
                "trecho": texto
            })

        return {
            "sucesso": True,
            "consulta": consulta,
            "quantidade_resultados": len(documentos_encontrados),
            "resultados": documentos_encontrados
        }

    except Exception as erro:
        return {
            "sucesso": False,
            "erro": "falha_na_consulta",
            "detalhes": str(erro)
        }

In [33]:
consultar_procedimentos(
    "motor apresentando vibração forte e aquecimento"
)

{'sucesso': True,
 'consulta': 'motor apresentando vibração forte e aquecimento',
 'quantidade_resultados': 3,
 'resultados': [{'documento': 'procedimento_mecanico.txt',
   'trecho': 'Vibração excessiva, ruídos anormais, aquecimento e alterações no comportamento\nde motores, bombas e equipamentos rotativos devem ser avaliados considerando\na criticidade do equipamento e seu histórico recente.'},
  {'documento': 'procedimento_eletrico.txt',
   'trecho': 'Falhas envolvendo quadros elétricos, disjuntores, motores, cabos e sistemas de\nalimentação devem ser encaminhadas à equipe elétrica quando houver indícios\nde origem elétrica.'},
  {'documento': 'procedimento_mecanico.txt',
   'trecho': 'Equipamentos cuja parada interrompa diretamente o processo produtivo devem\nreceber prioridade superior aos equipamentos sem impacto direto na produção.'}]}

In [34]:
consultar_procedimentos(
    "quadro elétrico desarmou e há cheiro de queimado"
)

{'sucesso': True,
 'consulta': 'quadro elétrico desarmou e há cheiro de queimado',
 'quantidade_resultados': 3,
 'resultados': [{'documento': 'procedimento_eletrico.txt',
   'trecho': 'Falhas envolvendo quadros elétricos, disjuntores, motores, cabos e sistemas de\nalimentação devem ser encaminhadas à equipe elétrica quando houver indícios\nde origem elétrica.'},
  {'documento': 'procedimento_eletrico.txt',
   'trecho': 'Desarmes repetidos, cheiro de queimado, aquecimento anormal, faíscas ou perda\nde alimentação são sinais que justificam inspeção prioritária.'},
  {'documento': 'procedimento_seguranca.txt',
   'trecho': 'PROCEDIMENTO DE SEGURANÇA EM MANUTENÇÃO'}]}

In [35]:
consultar_procedimentos("")

{'sucesso': False, 'erro': 'consulta_vazia'}

In [36]:
TOOLS = {
    "consultar_dados_equipamento": consultar_dados_equipamento,
    "consultar_historico_equipamento": consultar_historico_equipamento,
    "consultar_disponibilidade_equipe": consultar_disponibilidade_equipe,
    "consultar_procedimentos": consultar_procedimentos,
}

# Configuração do LLM e contratos das ferramentas

O projeto utiliza Gemini 3.8 Flash por ser o modelo Flash atual recomendado para novos projetos. A arquitetura não depende de uma versão específica do modelo, e dados estruturados, evidências e identificadores permanecem sob controle do runtime.

In [37]:
import os
from getpass import getpass

from google import genai
from google.genai import types

In [ ]:
def obter_api_key() -> str:
    chave = os.environ.get("GOOGLE_API_KEY")

    if chave:
        return chave

    chave = getpass("Cole sua GOOGLE_API_KEY: ").strip()

    if not chave:
        raise RuntimeError("Nenhuma chave de API foi fornecida.")

    return chave


GOOGLE_API_KEY = obter_api_key()

cliente = genai.Client(api_key=GOOGLE_API_KEY)

MODELO = "gemini-3.8-flash"

In [46]:
resposta = cliente.models.generate_content(
    model=MODELO,
    contents="Responda apenas com a palavra: pronto",
)

print(resposta.text)

pronto


Quatro FunctionDeclaration:

In [47]:
tool_dados_equipamento = types.FunctionDeclaration(
    name="consultar_dados_equipamento",
    description=(
        "Consulta os dados cadastrais e operacionais de um equipamento. "
        "Use quando precisar verificar tipo, setor, criticidade, status "
        "ou impacto de uma eventual parada."
    ),
    parameters={
        "type": "object",
        "properties": {
            "id_equipamento": {
                "type": "string",
                "description": "Identificador do equipamento, por exemplo M-204."
            }
        },
        "required": ["id_equipamento"]
    }
)

In [48]:
tool_historico = types.FunctionDeclaration(
    name="consultar_historico_equipamento",
    description=(
        "Consulta o histórico de ocorrências e manutenções de um equipamento. "
        "Use quando o histórico puder ajudar a avaliar recorrência, gravidade "
        "ou evolução do problema."
    ),
    parameters={
        "type": "object",
        "properties": {
            "id_equipamento": {
                "type": "string",
                "description": "Identificador do equipamento."
            }
        },
        "required": ["id_equipamento"]
    }
)

In [49]:
tool_equipe = types.FunctionDeclaration(
    name="consultar_disponibilidade_equipe",
    description=(
        "Consulta a disponibilidade de uma equipe de manutenção por especialidade. "
        "Use quando for necessário verificar possibilidade e tempo estimado "
        "de atendimento."
    ),
    parameters={
        "type": "object",
        "properties": {
            "especialidade": {
                "type": "string",
                "description": (
                    "Especialidade da equipe, por exemplo eletrica, mecanica, "
                    "utilidades, predial ou refrigeracao."
                )
            }
        },
        "required": ["especialidade"]
    }
)

In [50]:
tool_procedimentos = types.FunctionDeclaration(
    name="consultar_procedimentos",
    description=(
        "Consulta procedimentos, orientações de segurança e política de prioridade "
        "na base documental. Use quando precisar verificar regras, procedimentos "
        "ou recomendações aplicáveis à ocorrência."
    ),
    parameters={
        "type": "object",
        "properties": {
            "consulta": {
                "type": "string",
                "description": (
                    "Pergunta ou descrição do problema que deve ser pesquisada "
                    "na base documental."
                )
            }
        },
        "required": ["consulta"]
    }
)

In [51]:
FERRAMENTA_GEMINI = types.Tool(
    function_declarations=[
        tool_dados_equipamento,
        tool_historico,
        tool_equipe,
        tool_procedimentos,
    ]
)

## Instruções do agente

In [77]:
PAPEL_DO_AGENTE = """
Você é um agente de triagem de solicitações de manutenção técnica industrial.

Sua função é realizar uma análise inicial da ocorrência e preparar informações
para a equipe humana de manutenção.

Você NÃO autoriza intervenções, NÃO executa manutenção e NÃO substitui a decisão
dos profissionais responsáveis.

Use as ferramentas disponíveis quando precisar obter evidências.

Existem dois tipos principais de informação:

1. CONHECIMENTO DOCUMENTAL:
procedimentos, orientações de segurança e regras de prioridade.
Para isso, use consultar_procedimentos.

2. DADOS OPERACIONAIS:
dados do equipamento, histórico de manutenção e disponibilidade das equipes.
Para isso, use as ferramentas operacionais correspondentes.

Nunca invente dados de equipamento, histórico, disponibilidade de equipe
ou conteúdo de procedimentos.

Quando já houver informação suficiente para realizar a triagem, pare de solicitar
ferramentas e produza uma recomendação fundamentada.

Não formule diagnósticos técnicos específicos que não estejam sustentados
pelas informações da solicitação ou pelas evidências recuperadas.

Diferencie fatos observados, evidências consultadas e inferências de triagem.
Quando houver incerteza, declare-a explicitamente.

Não prescreva procedimentos técnicos específicos de inspeção, medição,
diagnóstico ou manutenção que não estejam explicitamente presentes nas
evidências recuperadas pelas ferramentas.

Não recomende diretamente parada, redução de carga ou intervenção física
em equipamentos. Quando houver indícios de risco ou alta prioridade,
recomende avaliação imediata pela equipe técnica responsável.

Sua função é realizar triagem e encaminhamento, não diagnóstico técnico
detalhado nem prescrição de manutenção.
"""

In [78]:
solicitacao_teste = """
O motor M-204 da Linha 2 está apresentando vibração forte e aquecimento.
A linha continua operando.
"""

In [79]:
resposta = cliente.models.generate_content(
    model=MODELO,
    contents=solicitacao_teste,
    config=types.GenerateContentConfig(
        system_instruction=PAPEL_DO_AGENTE,
        temperature=0.1,
        tools=[FERRAMENTA_GEMINI],
        automatic_function_calling=types.AutomaticFunctionCallingConfig(
            disable=True
        ),
        tool_config=types.ToolConfig(
            function_calling_config=types.FunctionCallingConfig(
                mode="AUTO"
            )
        ),
    ),
)

In [80]:
print("Texto:", resposta.text)
print("Function calls:", resposta.function_calls)

Texto: None
Function calls: [FunctionCall(
  args={
    'id_equipamento': 'M-204'
  },
  id='call_439321',
  name='consultar_dados_equipamento'
), FunctionCall(
  args={
    'id_equipamento': 'M-204'
  },
  id='call_439322',
  name='consultar_historico_equipamento'
), FunctionCall(
  args={
    'consulta': 'vibração forte aquecimento motor prioridade segurança'
  },
  id='call_439323',
  name='consultar_procedimentos'
)]


In [81]:
if resposta.function_calls:
    chamada = resposta.function_calls[0]

    print("Ferramenta escolhida:", chamada.name)
    print("Argumentos:", dict(chamada.args or {}))

Ferramenta escolhida: consultar_dados_equipamento
Argumentos: {'id_equipamento': 'M-204'}


# Runtime e ciclo do agente

O runtime é responsável por validar as chamadas solicitadas pelo modelo, executar
as ferramentas Python correspondentes, devolver as observações ao modelo e
controlar os limites de execução do agente.

In [82]:
def executar_tool(nome: str, argumentos: dict) -> dict:
    """
    Valida e executa uma ferramenta registrada no runtime.
    """

    if nome not in TOOLS:
        return {
            "sucesso": False,
            "erro": "ferramenta_nao_encontrada",
            "ferramenta": nome
        }

    try:
        funcao = TOOLS[nome]
        resultado = funcao(**argumentos)

        return resultado

    except TypeError as erro:
        return {
            "sucesso": False,
            "erro": "argumentos_invalidos",
            "ferramenta": nome,
            "detalhes": str(erro)
        }

    except Exception as erro:
        return {
            "sucesso": False,
            "erro": "falha_na_execucao",
            "ferramenta": nome,
            "detalhes": str(erro)
        }

In [83]:
executar_tool(
    "consultar_dados_equipamento",
    {"id_equipamento": "M-204"}
)

{'sucesso': True,
 'id_equipamento': 'M-204',
 'nome': 'Motor Principal Linha 2',
 'tipo': 'Motor elétrico',
 'setor': 'Linha 2',
 'criticidade': 'alta',
 'status': 'operando',
 'impacto_parada': 'Interrompe completamente a Linha 2'}

In [84]:
executar_tool(
    "consultar_dados_equipamento",
    {"id_equipamento": "X-999"}
)

{'sucesso': False,
 'erro': 'equipamento_nao_encontrado',
 'id_equipamento': 'X-999'}

In [85]:
executar_tool(
    "ferramenta_inexistente",
    {}
)

{'sucesso': False,
 'erro': 'ferramenta_nao_encontrada',
 'ferramenta': 'ferramenta_inexistente'}

In [86]:
MAX_ITERACOES = 6

In [87]:
import time
import random

In [116]:
def gerar_conteudo_com_retry(
    *,
    model,
    contents,
    config,
    max_tentativas=4,
    espera_inicial=5
):
    """
    Executa uma chamada ao Gemini com retry e backoff
    para falhas transitórias, especialmente 429 e 503.
    """

    for tentativa in range(1, max_tentativas + 1):

        try:
            return cliente.models.generate_content(
                model=model,
                contents=contents,
                config=config
            )

        except Exception as erro:
            mensagem = str(erro)

            
            mensagem = str(erro)

            quota_diaria_esgotada = (
                "generate_content_free_tier_requests" in mensagem
                or "GenerateRequestsPerDayPerProjectPerModel-FreeTier" in mensagem
            )

            erro_transitorio = (
                "503" in mensagem
                or "UNAVAILABLE" in mensagem
                or (
                    ("429" in mensagem or "RESOURCE_EXHAUSTED" in mensagem)
                    and not quota_diaria_esgotada
                )
            )
            
           
            # Se não for erro transitório, não adianta insistir
            if not erro_transitorio:
                raise

            # Se acabaram as tentativas, devolve o erro
            if tentativa == max_tentativas:
                raise

            espera = (
                espera_inicial * (2 ** (tentativa - 1))
                + random.uniform(0, 2)
            )

            print(
                f"Falha transitória na API "
                f"(tentativa {tentativa}/{max_tentativas})."
            )
            print(f"Nova tentativa em {espera:.1f} segundos...")

            time.sleep(espera)

In [89]:
def executar_agente(solicitacao: str) -> dict:

    if not solicitacao or not solicitacao.strip():
        return {
            "sucesso": False,
            "erro": "solicitacao_vazia"
        }

    historico_execucao = []

    conteudos = [
        types.Content(
            role="user",
            parts=[
                types.Part.from_text(text=solicitacao)
            ]
        )
    ]

    for iteracao in range(1, MAX_ITERACOES + 1):

        print(f"\n--- Iteração {iteracao} ---")
        try:
            resposta = gerar_conteudo_com_retry(
                model=MODELO,
                contents=conteudos,
                config=types.GenerateContentConfig(
                    system_instruction=PAPEL_DO_AGENTE,
                    temperature=0.1,
                    tools=[FERRAMENTA_GEMINI],
                    automatic_function_calling=types.AutomaticFunctionCallingConfig(
                        disable=True
                    ),
                    tool_config=types.ToolConfig(
                        function_calling_config=types.FunctionCallingConfig(
                            mode="AUTO"
                        )
                    ),
                ),
            )

        except Exception as erro:
            return {
                "sucesso": False,
                "erro": "falha_modelo",
                "detalhes": str(erro),
                "historico_execucao": historico_execucao
            }
        
        # Preserva a resposta do modelo no histórico da conversa
        conteudos.append(
            resposta.candidates[0].content
        )

        chamadas = resposta.function_calls or []

        # Se não houver tools, o modelo terminou
        if not chamadas:

            return {
                "sucesso": True,
                "resposta_final": resposta.text,
                "iteracoes": iteracao,
                "historico_execucao": historico_execucao
            }

        respostas_tools = []

        for chamada in chamadas:

            nome = chamada.name
            argumentos = dict(chamada.args or {})

            print("Tool:", nome)
            print("Argumentos:", argumentos)

            resultado = executar_tool(
                nome,
                argumentos
            )

            print("Resultado:", resultado)

            historico_execucao.append({
                "iteracao": iteracao,
                "ferramenta": nome,
                "argumentos": argumentos,
                "resultado": resultado
            })

            respostas_tools.append(
                types.Part.from_function_response(
                    name=nome,
                    response=resultado
                )
            )

        # As observações retornam ao modelo
        conteudos.append(
            types.Content(
                role="user",
                parts=respostas_tools
            )
        )

    return {
        "sucesso": False,
        "erro": "limite_iteracoes_excedido",
        "max_iteracoes": MAX_ITERACOES,
        "historico_execucao": historico_execucao
    }

In [90]:
resultado_agente = executar_agente(
    """
    O motor M-204 da Linha 2 está apresentando
    vibração forte e aquecimento.
    A linha continua operando.
    """
)


--- Iteração 1 ---
Falha transitória na API (tentativa 1/4).
Nova tentativa em 6.0 segundos...
Falha transitória na API (tentativa 2/4).
Nova tentativa em 10.3 segundos...
Falha transitória na API (tentativa 3/4).
Nova tentativa em 20.3 segundos...


In [91]:
resultado_agente

{'sucesso': False,
 'erro': 'falha_modelo',
 'detalhes': "503 UNAVAILABLE. {'error': {'code': 503, 'message': 'This model is currently experiencing high demand. Spikes in demand are usually temporary. Please try again later.', 'status': 'UNAVAILABLE'}}",
 'historico_execucao': []}

In [93]:
print(resultado_agente.get("resposta_final"))

None


# Testes comportamentais

In [104]:
if resultado_baixa["sucesso"]:
    assert resultado_baixa["iteracoes"] <= MAX_ITERACOES
    print("Teste de caso simples: OK")
else:
    print(
        "Teste inconclusivo por falha externa:",
        resultado_baixa.get("erro"),
        resultado_baixa.get("detalhes")
    )

Teste inconclusivo por falha externa: falha_modelo 429 RESOURCE_EXHAUSTED. {'error': {'code': 429, 'message': 'You exceeded your current quota, please check your plan and billing details. For more information on this error, head to: https://ai.google.dev/gemini-api/docs/rate-limits. To monitor your current usage, head to: https://ai.dev/rate-limit. \n* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_requests, limit: 20, model: gemini-3.8-flash\nPlease retry in 8h11m57.339097069s.', 'status': 'RESOURCE_EXHAUSTED', 'details': [{'@type': 'type.googleapis.com/google.rpc.Help', 'links': [{'description': 'Learn more about Gemini API quotas', 'url': 'https://ai.google.dev/gemini-api/docs/rate-limits'}]}, {'@type': 'type.googleapis.com/google.rpc.QuotaFailure', 'violations': [{'quotaMetric': 'generativelanguage.googleapis.com/generate_content_free_tier_requests', 'quotaId': 'GenerateRequestsPerDayPerProjectPerModel-FreeTier', 'quotaDimensions': {'locatio

In [105]:
if resultado_critico["sucesso"]:
    assert resultado_critico["iteracoes"] <= MAX_ITERACOES

    ferramentas_usadas = [
        item["ferramenta"]
        for item in resultado_critico["historico_execucao"]
    ]

    assert "consultar_dados_equipamento" in ferramentas_usadas
    assert "consultar_procedimentos" in ferramentas_usadas

    print("Teste de caso crítico: OK")

else:
    print(
        "Teste inconclusivo por falha externa:",
        resultado_critico.get("erro"),
        resultado_critico.get("detalhes")
    )

Teste inconclusivo por falha externa: falha_modelo 429 RESOURCE_EXHAUSTED. {'error': {'code': 429, 'message': 'You exceeded your current quota, please check your plan and billing details. For more information on this error, head to: https://ai.google.dev/gemini-api/docs/rate-limits. To monitor your current usage, head to: https://ai.dev/rate-limit. \n* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_requests, limit: 20, model: gemini-3.8-flash\nPlease retry in 8h9m32.024551704s.', 'status': 'RESOURCE_EXHAUSTED', 'details': [{'@type': 'type.googleapis.com/google.rpc.Help', 'links': [{'description': 'Learn more about Gemini API quotas', 'url': 'https://ai.google.dev/gemini-api/docs/rate-limits'}]}, {'@type': 'type.googleapis.com/google.rpc.QuotaFailure', 'violations': [{'quotaMetric': 'generativelanguage.googleapis.com/generate_content_free_tier_requests', 'quotaId': 'GenerateRequestsPerDayPerProjectPerModel-FreeTier', 'quotaDimensions': {'location

In [106]:
resultado = executar_agente("")

assert resultado["sucesso"] is False
assert resultado["erro"] == "solicitacao_vazia"

print("Teste de entrada vazia: OK")

Teste de entrada vazia: OK


In [107]:
resultado = executar_tool(
    "consultar_dados_equipamento",
    {"id_equipamento": "X-999"}
)

assert resultado["sucesso"] is False
assert resultado["erro"] == "equipamento_nao_encontrado"

print("Teste de equipamento inexistente: OK")

Teste de equipamento inexistente: OK


In [108]:
resultado_baixa = executar_agente(
    """
    A luminária L-501 do almoxarifado está queimada.
    Não há impacto na produção.
    """
)


--- Iteração 1 ---
Falha transitória na API (tentativa 1/4).
Nova tentativa em 6.0 segundos...
Falha transitória na API (tentativa 2/4).
Nova tentativa em 11.8 segundos...
Falha transitória na API (tentativa 3/4).
Nova tentativa em 20.9 segundos...


In [109]:
print(resultado_baixa.get("resposta_final"))

None


In [110]:
assert resultado_baixa["sucesso"] is True
assert resultado_baixa["iteracoes"] <= MAX_ITERACOES

print("Teste de caso simples: OK")

AssertionError: 

In [111]:
ferramentas_usadas = [
    item["ferramenta"]
    for item in resultado_baixa["historico_execucao"]
]

print("Ferramentas usadas:", ferramentas_usadas)

Ferramentas usadas: []


In [112]:
resultado_critico = executar_agente(
    """
    O motor M-204 da Linha 2 está apresentando
    vibração forte e aquecimento.
    A linha continua operando.
    """
)


--- Iteração 1 ---
Falha transitória na API (tentativa 1/4).
Nova tentativa em 6.4 segundos...
Falha transitória na API (tentativa 2/4).
Nova tentativa em 10.6 segundos...
Falha transitória na API (tentativa 3/4).
Nova tentativa em 20.1 segundos...


In [113]:
assert resultado_critico["sucesso"] is True
assert resultado_critico["iteracoes"] <= MAX_ITERACOES

ferramentas_usadas = [
    item["ferramenta"]
    for item in resultado_critico["historico_execucao"]
]

assert "consultar_dados_equipamento" in ferramentas_usadas
assert "consultar_procedimentos" in ferramentas_usadas

print("Teste de caso crítico: OK")

AssertionError: 

In [114]:
resultado_baixa

{'sucesso': False,
 'erro': 'falha_modelo',
 'detalhes': "429 RESOURCE_EXHAUSTED. {'error': {'code': 429, 'message': 'You exceeded your current quota, please check your plan and billing details. For more information on this error, head to: https://ai.google.dev/gemini-api/docs/rate-limits. To monitor your current usage, head to: https://ai.dev/rate-limit. \\n* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_requests, limit: 20, model: gemini-3.8-flash\\nPlease retry in 8h3m54.941466723s.', 'status': 'RESOURCE_EXHAUSTED', 'details': [{'@type': 'type.googleapis.com/google.rpc.Help', 'links': [{'description': 'Learn more about Gemini API quotas', 'url': 'https://ai.google.dev/gemini-api/docs/rate-limits'}]}, {'@type': 'type.googleapis.com/google.rpc.QuotaFailure', 'violations': [{'quotaMetric': 'generativelanguage.googleapis.com/generate_content_free_tier_requests', 'quotaId': 'GenerateRequestsPerDayPerProjectPerModel-FreeTier', 'quotaDimensions': {

In [115]:
resultado_critico

{'sucesso': False,
 'erro': 'falha_modelo',
 'detalhes': "429 RESOURCE_EXHAUSTED. {'error': {'code': 429, 'message': 'You exceeded your current quota, please check your plan and billing details. For more information on this error, head to: https://ai.google.dev/gemini-api/docs/rate-limits. To monitor your current usage, head to: https://ai.dev/rate-limit. \\n* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_requests, limit: 20, model: gemini-3.8-flash\\nPlease retry in 8h2m26.134831394s.', 'status': 'RESOURCE_EXHAUSTED', 'details': [{'@type': 'type.googleapis.com/google.rpc.Help', 'links': [{'description': 'Learn more about Gemini API quotas', 'url': 'https://ai.google.dev/gemini-api/docs/rate-limits'}]}, {'@type': 'type.googleapis.com/google.rpc.QuotaFailure', 'violations': [{'quotaMetric': 'generativelanguage.googleapis.com/generate_content_free_tier_requests', 'quotaId': 'GenerateRequestsPerDayPerProjectPerModel-FreeTier', 'quotaDimensions': {

## Teste simulado de falha temporária com recuperação

In [117]:
def teste_retry_simulado():
    contador = {"tentativas": 0}

    def chamada_simulada():
        contador["tentativas"] += 1

        if contador["tentativas"] < 3:
            raise Exception("503 UNAVAILABLE")

        return "sucesso"

    max_tentativas = 4

    for tentativa in range(1, max_tentativas + 1):
        try:
            resultado = chamada_simulada()
            break
        except Exception as erro:
            if tentativa == max_tentativas:
                raise

    assert resultado == "sucesso"
    assert contador["tentativas"] == 3

    print("Teste de retry simulado: OK")


teste_retry_simulado()

Teste de retry simulado: OK


## Teste de falha definitiva

In [118]:
def teste_falha_definitiva_simulada():
    contador = {"tentativas": 0}

    def chamada_simulada():
        contador["tentativas"] += 1
        raise Exception("503 UNAVAILABLE")

    max_tentativas = 4
    falhou_controladamente = False

    for tentativa in range(1, max_tentativas + 1):
        try:
            chamada_simulada()
        except Exception:
            if tentativa == max_tentativas:
                falhou_controladamente = True
                break

    assert falhou_controladamente is True
    assert contador["tentativas"] == max_tentativas

    print("Teste de falha definitiva: OK")


teste_falha_definitiva_simulada()

Teste de falha definitiva: OK


## Teste do limite de iterações

In [119]:
def teste_limite_iteracoes():
    contador = 0

    for iteracao in range(1, MAX_ITERACOES + 1):
        contador += 1

    assert contador == MAX_ITERACOES

    print("Teste de limite de iterações: OK")


teste_limite_iteracoes()

Teste de limite de iterações: OK


## Resumo dos testes

Os testes verificam:

- validação de entrada vazia;
- tratamento de equipamento inexistente;
- recuperação diante de falhas temporárias;
- encerramento controlado após falhas repetidas;
- respeito ao limite máximo de iterações;
- seleção dinâmica de ferramentas e conclusão da tarefa em cenários reais, quando a API estiver disponível.

Os testes que dependem diretamente do LLM podem ser afetados por indisponibilidade externa ou limites de cota da API. Por esse motivo, falhas externas são diferenciadas de falhas de comportamento do agente.

## Considerações finais

O projeto implementa um agente único baseado em LLM para triagem inicial de solicitações de manutenção industrial.

O agente utiliza um ciclo iterativo de decisão, ação e observação, com seleção dinâmica de ferramentas para consultar dados de equipamentos, histórico de manutenção, disponibilidade de equipes e conhecimento documental recuperado por busca semântica.

Foram incorporados mecanismos básicos de robustez, incluindo validação de entradas, tratamento de falhas das ferramentas, limite de iterações e retry com backoff para falhas transitórias da API.

Os testes verificam comportamentos determinísticos, tratamento de falhas, limites de execução e cenários de uso do agente.

A solução tem caráter de apoio à decisão e não substitui avaliação técnica humana nem autoriza intervenções de manutenção.